In [5]:
#Task 1 — Design a Class

class TrainingRun:
    run_count = 0
    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate
        self._status = "pending"
        TrainingRun.run_count += 1
    def start(self):
        self._status = "running"
        print(f"{self.model_name} training started.")
    def summary(self):
        print(
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self._status}"
        )
    @classmethod
    def from_config(cls, config_dict):
        return cls(
            config_dict["model_name"],
            config_dict["learning_rate"]
        )
    @staticmethod
    def validate_learning_rate(lr):
        return 0 < lr < 1

#Test

run1 = TrainingRun("GPT-Mini", 0.01)
run2 = TrainingRun("BERT", 0.02)
run3 = TrainingRun("ResNet", 0.001)
run1.start()
run1.summary()
config = {
    "model_name": "Llama",
    "learning_rate": 0.05
}
run4 = TrainingRun.from_config(config)
run4.summary()
print("Run count:", TrainingRun.run_count)

GPT-Mini training started.
Model: GPT-Mini, Learning Rate: 0.01, Status: running
Model: Llama, Learning Rate: 0.05, Status: pending
Run count: 4


In [6]:
#Task 2 — Encapsulation

class TrainingRun:
    run_count = 0
    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate
        self._status = "pending"
        self.__api_key = "fake-token-123"
        TrainingRun.run_count += 1
    @property
    def status(self):
        return self._status
    @status.setter
    def status(self, value):
        if value not in ["pending", "running", "done"]:
            raise ValueError("Status must be pending, running, or done")
        self._status = value

#Test

run = TrainingRun("GPT-Mini", 0.01)
run.status = "running"
print("Status:", run.status)
try:
    run.status = "paused"
except ValueError as e:
    print("Error:", e)
try:
    print(run.__api_key)
except AttributeError:
    print("Direct access to __api_key is not allowed")
print("Name-mangled API key:", run._TrainingRun__api_key)

Status: running
Error: Status must be pending, running, or done
Direct access to __api_key is not allowed
Name-mangled API key: fake-token-123


In [13]:
#Task 3 — Inheritanceclass TrainingRun:
    
class TrainingRun:
    run_count = 0
    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate
        self._status = "pending"
        TrainingRun.run_count += 1
    def summary(self):
        print(
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self._status}"
        )
class LRSchedulerRun(TrainingRun):
    def __init__(self, model_name, learning_rate, schedule):
        super().__init__(model_name, learning_rate)
        self.schedule = schedule
    def summary(self):
        super().summary()
        print("Schedule:", self.schedule)
run = LRSchedulerRun(
    "GPT-Mini",
    0.01,
    [0.01, 0.005, 0.001]
)
run.summary()
print("Total runs:", TrainingRun.run_count)

Model: GPT-Mini, Learning Rate: 0.01, Status: pending
Schedule: [0.01, 0.005, 0.001]
Total runs: 1


In [14]:
#Task 4 - Polymorphism

class TrainingRun:
    run_count = 0
    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate
        self._status = "pending"
        TrainingRun.run_count += 1
    def summary(self):
        print(
            f"Model: {self.model_name}, "
            f"Learning Rate: {self.learning_rate}, "
            f"Status: {self._status}"
        )
class LRSchedulerRun(TrainingRun):
    def __init__(self, model_name, learning_rate, schedule):
        super().__init__(model_name, learning_rate)
        self.schedule = schedule
    def summary(self):
        super().summary()
        print("Schedule:", self.schedule)
class EarlyStoppingRun(TrainingRun):
    def __init__(self, model_name, learning_rate, patience):
        super().__init__(model_name, learning_rate)
        self.patience = patience
    def summary(self):
        super().summary()
        print("Patience:", self.patience)
def print_all_summaries(runs):
    for run in runs:
        run.summary()
run1 = TrainingRun("GPT-Mini", 0.01)
run2 = LRSchedulerRun(
    "BERT",
    0.02,
    [0.02, 0.01, 0.005]
)
run3 = EarlyStoppingRun(
    "ResNet",
    0.001,
    5
)
runs = [run1, run2, run3]
print_all_summaries(runs)

Model: GPT-Mini, Learning Rate: 0.01, Status: pending
Model: BERT, Learning Rate: 0.02, Status: pending
Schedule: [0.02, 0.01, 0.005]
Model: ResNet, Learning Rate: 0.001, Status: pending
Patience: 5


In [18]:
#Task 5 - Duck Typing vs Interfaces

#Part A — Duck Typing
class Cleaner:
    def process(self, data):
        return data.strip()
class Tokenizer:
    def process(self, data):
        return data.split()
class Normalizer:
    def process(self, data):
        return [word.lower() for word in data]
def run_pipeline(steps, data):
    for step in steps:
        data = step.process(data)
    return data
steps = [Cleaner(), Tokenizer(), Normalizer()]
result = run_pipeline(
    steps,
    "  Hello Python World  "
)
print(result)

#Part B — ABC
from abc import ABC, abstractmethod
class Step(ABC):
    @abstractmethod
    def process(self, data):
        pass
class Cleaner(Step):
    def process(self, data):
        return data.strip()
class Tokenizer(Step):
    def process(self, data):
        return data.split()
class Normalizer(Step):
    def process(self, data):
        return [word.lower() for word in data]
def run_pipeline(steps, data):
    for step in steps:
        data = step.process(data)
    return data
steps = [
    Cleaner(),
    Tokenizer(),
    Normalizer()
]
result = run_pipeline(
    steps,
    "  Hello Python World  "
)
print(result)
try:
    Step()
except TypeError:
    print("Step cannot be instantiated directly")

['hello', 'python', 'world']
['hello', 'python', 'world']
Step cannot be instantiated directly


In [20]:
#Task 6 - Dunder Methods

class TrainingRun:
    def __init__(self, model_name, learning_rate):
        self.model_name = model_name
        self.learning_rate = learning_rate
    def __str__(self):
        return f"TrainingRun(model='{self.model_name}', lr={self.learning_rate})"
    def __eq__(self, other):
        if not isinstance(other, TrainingRun):
            return NotImplemented
        return (
            self.model_name == other.model_name
            and self.learning_rate == other.learning_rate
        )
run1 = TrainingRun("gpt-mini", 0.01)
run2 = TrainingRun("gpt-mini", 0.01)
run3 = TrainingRun("bert", 0.02)
print(run1)
print(run1 == run2)
print(run1 == run3)

TrainingRun(model='gpt-mini', lr=0.01)
True
False
